# 光线传媒（300251.SZ）近一年股价分析

---

**项目概述**

本 Notebook 展示从 **Tushare** 获取 A 股光线传媒（300251.SZ）近一年（2025-07-01 ~ 2026-07-01）的日线行情数据，并进行数据清洗与可视化：

- 收盘价趋势图 + 移动均线
- K 线图（蜡烛图）+ 成交量柱状图

---

**所需库**

- `tushare` — A股数据接口
- `pandas` — 数据处理
- `matplotlib` — 基础绘图
- `mplfinance` — 金融 K 线图专用库


In [ ]:
# ============================
# Step 1: 导入依赖库
# ============================

import tushare as ts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import mplfinance as mpf
import json
import warnings
warnings.filterwarnings('ignore')

# 设置 matplotlib 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 所有库导入成功")
print(f"  pandas 版本: {pd.__version__}")
print(f"  tushare 版本: {ts.__version__}")


## Step 2: 通过 Tushare API 获取数据

使用 `pro.daily()` 接口获取 300251.SZ 的日线行情。

> 💡 需要 Tushare Token（120+ 积分才能访问 `daily` 接口）

> 📍 数据文件已保存为 `光线传媒_300251_daily.json`，如果无法连接 Tushare，后续 Cell 可直接从本地加载。


In [ ]:
# ============================
# Step 2: Tushare 获取日线数据
# ============================

# ======== 请在此填入你的 Tushare Token ========
MY_TOKEN = "你的_Tushare_Token"
# =============================================

try:
    # 初始化 pro 接口
    pro = ts.pro_api(MY_TOKEN)

    # 获取 2025-07-01 到 2026-07-01 的日线数据
    df = pro.daily(ts_code='300251.SZ', start_date='20250701', end_date='20260701')

    print(f"✅ 从 Tushare 获取了 {len(df)} 条记录")
    print(f"  字段: {list(df.columns)}")
    print()

    # 按日期升序排列
    df = df.sort_values('trade_date').reset_index(drop=True)

    # 重命名列，便于后续使用
    df.rename(columns={
        'trade_date': 'date',
        'vol': 'volume',
        'pct_chg': 'pct_change'
    }, inplace=True)

    # 转换日期格式
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

    print("📋 数据预览（前 5 行）：")
    display(df.head())

    print(f"\n📊 数据概览：")
    print(f"  时间范围: {df['date'].min().strftime('%Y-%m-%d')} → {df['date'].max().strftime('%Y-%m-%d')}")
    print(f"  交易天数: {len(df)}")
    print(f"  最高价: ¥{df['high'].max():.2f}")
    print(f"  最低价: ¥{df['low'].min():.2f}")
    print(f"  区间涨跌: {(df.iloc[-1]['close'] / df.iloc[0]['close'] - 1) * 100:.2f}%")

except Exception as e:
    print(f"⚠️ Tushare 连接失败: {e}")
    print("💡 请检查 Token 是否正确、积分是否达标")
    print("   → 可运行下一个 Cell，从本地 JSON 文件加载数据")


## Step 2（备选）: 从本地 JSON 文件加载

如果无法连接 Tushare，直接加载已保存的数据文件。


In [ ]:
# ============================
# Step 2 备选: 从本地 JSON 加载数据
# ============================

import json

with open('光线传媒_300251_daily.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"📋 从本地加载了 {len(df)} 条记录")
print(f"  时间范围: {df['date'].min().strftime('%Y-%m-%d')} → {df['date'].max().strftime('%Y-%m-%d')}")
display(df.head())

# 统一字段名（如果从 tushare 获取则字段名不同）
if 'volume' not in df.columns and 'vol' in df.columns:
    df.rename(columns={'vol': 'volume', 'pct_chg': 'pct_change'}, inplace=True)
elif 'volume' not in df.columns and 'amount' in df.columns:
    # 从本地 JSON 加载的数据已标准化
    pass    # 本地 JSON 中 volume = 成交量（手）, amount = 成交额（千元）


## Step 3: 数据清洗与特征工程

计算移动均线等衍生指标。


In [ ]:
# ============================
# Step 3: 数据清洗与特征工程
# ============================

# 检查缺失值
print("🔍 缺失值统计：")
print(df.isnull().sum())

# 计算移动均线
df['MA5']  = df['close'].rolling(window=5).mean()
df['MA10'] = df['close'].rolling(window=10).mean()
df['MA20'] = df['close'].rolling(window=20).mean()
df['MA60'] = df['close'].rolling(window=60).mean()

# 成交量单位转换（手 → 万手）
df['volume_wan'] = df['volume'] / 10000

# 计算涨跌颜色（用于 K 线图）
df['change'] = df['close'] - df['open']

print("\n✅ 特征工程完成 — 已计算 MA5 / MA10 / MA20 / MA60")
display(df[['date', 'open', 'close', 'high', 'low', 'volume_wan', 'MA5', 'MA20', 'MA60']].tail(10))


## Step 4: 收盘价趋势图

绘制每日收盘价 + MA5 / MA20 均线。


In [ ]:
# ============================
# Step 4: 收盘价趋势图
# ============================

fig, ax = plt.subplots(figsize=(16, 7))

# 收盘价曲线
ax.plot(df['date'], df['close'], color='#e74c3c', linewidth=1.5, label='收盘价', alpha=0.9)
ax.fill_between(df['date'], df['close'], df['close'].min() * 0.95,
                color='#e74c3c', alpha=0.08)

# 移动均线
ax.plot(df['date'], df['MA5'],  color='#3498db', linewidth=1, label='MA5',  alpha=0.8)
ax.plot(df['date'], df['MA20'], color='#f39c12', linewidth=1.2, linestyle='--', label='MA20', alpha=0.9)
ax.plot(df['date'], df['MA60'], color='#9b59b6', linewidth=1, linestyle='-.', label='MA60', alpha=0.7)

# 标注最高点和最低点
max_idx = df['close'].idxmax()
min_idx = df['close'].idxmin()
ax.annotate(f'最高 ¥{df.loc[max_idx, "close"]:.2f}',
            xy=(df.loc[max_idx, 'date'], df.loc[max_idx, 'close']),
            xytext=(30, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red', fontweight='bold')
ax.annotate(f'最低 ¥{df.loc[min_idx, "close"]:.2f}',
            xy=(df.loc[min_idx, 'date'], df.loc[min_idx, 'close']),
            xytext=(30, -30), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=10, color='green', fontweight='bold')

# 格式化
ax.set_title('光线传媒（300251.SZ）收盘价趋势图', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('日期', fontsize=12)
ax.set_ylabel('价格（元）', fontsize=12)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

# 添加区间统计信息
change_pct = (df.iloc[-1]['close'] / df.iloc[0]['close'] - 1) * 100
stats_text = (
    f"区间: {df['date'].min().strftime('%Y.%m.%d')} → {df['date'].max().strftime('%Y.%m.%d')}\n"
    f"起点: ¥{df.iloc[0]['close']:.2f}  →  终点: ¥{df.iloc[-1]['close']:.2f}  |  "
    f"涨跌: {change_pct:+.2f}%  |  "
    f"最高: ¥{df['high'].max():.2f}  |  "
    f"最低: ¥{df['low'].min():.2f}"
)
plt.figtext(0.5, 0.01, stats_text, ha='center', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', edgecolor='#ddd'))

plt.tight_layout()
plt.show()

print(f"\n📊 区间统计:")
print(f"  起始收盘价: ¥{df.iloc[0]['close']:.2f}")
print(f"  最新收盘价: ¥{df.iloc[-1]['close']:.2f}")
print(f"  区间涨跌幅: {change_pct:+.2f}%")
print(f"  历史最高:   ¥{df['high'].max():.2f}  ({df.loc[df['high'].idxmax(), 'date'].strftime('%Y-%m-%d')})")
print(f"  历史最低:   ¥{df['low'].min():.2f}  ({df.loc[df['low'].idxmin(), 'date'].strftime('%Y-%m-%d')})")


## Step 5: 每日成交量柱状图

涨红跌绿，符合 A 股习惯。


In [ ]:
# ============================
# Step 5: 成交量柱状图
# ============================

fig, ax = plt.subplots(figsize=(16, 5))

# 按涨跌分色
colors = ['#e74c3c' if row['change'] >= 0 else '#27ae60' for _, row in df.iterrows()]

ax.bar(df['date'], df['volume_wan'], color=colors, width=1.0, alpha=0.8, edgecolor='none')

# 成交量均线
vol_ma20 = df['volume_wan'].rolling(20).mean()
ax.plot(df['date'], vol_ma20, color='#f39c12', linewidth=1.2, label='VOL_MA20', alpha=0.8)

ax.set_title('光线传媒（300251.SZ）每日成交量', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('日期', fontsize=11)
ax.set_ylabel('成交量（万手）', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

# 标注成交天量
max_vol_idx = df['volume_wan'].idxmax()
ax.annotate(f'天量 {df.loc[max_vol_idx, "volume_wan"]:.0f}万手',
            xy=(df.loc[max_vol_idx, 'date'], df.loc[max_vol_idx, 'volume_wan']),
            xytext=(20, 15), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='#c0392b'),
            fontsize=9, color='#c0392b', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 成交量统计:")
print(f"  日均成交量: {df['volume_wan'].mean():.0f} 万手")
print(f"  最大成交量: {df['volume_wan'].max():.0f} 万手 ({df.loc[max_vol_idx, 'date'].strftime('%Y-%m-%d')})")
print(f"  最小成交量: {df['volume_wan'].min():.0f} 万手")


## Step 6: K 线图 + 成交量（mplfinance）

使用 `mplfinance` 绘制专业金融 K 线图，包含：
- 日 K 蜡烛图
- MA5 / MA10 / MA20 / MA60 四根均线
- 底部成交量柱状图


In [ ]:
# ============================
# Step 6: mplfinance K线图 + 成交量
# ============================

# mplfinance 需要 Date 作为 index，且列名为 Open/High/Low/Close/Volume
df_mpf = df[['date', 'open', 'high', 'low', 'close', 'volume_wan']].copy()
df_mpf.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
df_mpf.set_index('Date', inplace=True)

# A 股配色：涨红跌绿
mc = mpf.make_marketcolors(
    up='#e74c3c',      # 涨 → 红色
    down='#27ae60',    # 跌 → 绿色
    edge='inherit',
    volume={'up': '#e74c3c', 'down': '#27ae60'},
    wick='inherit',
    alpha=0.85
)
style = mpf.make_mpf_style(
    marketcolors=mc,
    gridcolor='#e0e0e0',
    facecolor='white',
    figcolor='white',
    gridstyle='--',
    gridaxis='horizontal'
)

# 绘制
fig, axes = mpf.plot(
    df_mpf,
    type='candle',
    style=style,
    mav=(5, 10, 20, 60),
    volume=axes[12:] if 'axes' in dir() else True,  # 带成交量子图 — 直接用 volume=True
    title='光线传媒（300251.SZ）日K线图',
    ylabel='价格（元）',
    volume_ylabel='成交量（万手）',
    figsize=(16, 9),
    datetime_format='%Y-%m',
    xrotation=30,
    panel_ratios=(3, 1),
    returnfig=True,
    warn_too_much_data=250,
    update_width_config=dict(candle_linewidth=0.8)
)

fig.suptitle('光线传媒（300251.SZ）近一年日K线 + 成交量', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## Step 7: 综合看板（Matplotlib 多子图）

不依赖 mplfinance 的备选方案，用纯 matplotlib 实现 K 线 + 成交量联动图。


In [ ]:
# ============================
# Step 7: 纯 matplotlib 综合看板
# ============================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10),
                                 gridspec_kw={'height_ratios': [2.5, 1], 'hspace': 0.08},
                                 sharex=True)

# ---------- 上方：K 线蜡烛图 ----------
width = 0.6
for i, (_, row) in enumerate(df.iterrows()):
    color = '#e74c3c' if row['close'] >= row['open'] else '#27ae60'
    # 影线
    ax1.plot([i, i], [row['low'], row['high']], color=color, linewidth=0.8)
    # 实体
    body_bottom = min(row['open'], row['close'])
    body_height = abs(row['close'] - row['open'])
    ax1.bar(i, body_height, width, bottom=body_bottom, color=color, alpha=0.9, edgecolor=color, linewidth=0.3)

# 均线
date_indices = np.arange(len(df))
ax1.plot(date_indices, df['MA5'],  color='#3498db', linewidth=0.8, label='MA5')
ax1.plot(date_indices, df['MA10'], color='#2ecc71', linewidth=0.8, label='MA10')
ax1.plot(date_indices, df['MA20'], color='#f39c12', linewidth=1.0, label='MA20')
ax1.plot(date_indices, df['MA60'], color='#9b59b6', linewidth=0.8, label='MA60')

ax1.set_title('光线传媒（300251.SZ）日K线 + 成交量综合看板', fontsize=16, fontweight='bold')
ax1.set_ylabel('价格（元）', fontsize=12)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.25, axis='y')
ax1.set_xlim(-3, len(df) + 3)

# 用日期格式化 x 轴
tick_positions = np.arange(0, len(df), max(1, len(df) // 8))
tick_labels = [df.iloc[i]['date'].strftime('%Y-%m') for i in tick_positions]
ax1.set_xticks([])

# ---------- 下方：成交量柱状图 ----------
colors_vol = ['#e74c3c' if row['close'] >= row['open'] else '#27ae60' for _, row in df.iterrows()]
ax2.bar(date_indices, df['volume_wan'], color=colors_vol, width=1.0, alpha=0.75, edgecolor='none')

# 成交量 MA20
ax2.plot(date_indices, df['volume_wan'].rolling(20).mean(),
         color='#f39c12', linewidth=1, label='VOL_MA20')

ax2.set_xlabel('日期', fontsize=12)
ax2.set_ylabel('成交量\n（万手）', fontsize=12)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.25, axis='y')
ax2.set_xlim(-3, len(df) + 3)
ax2.set_xticks(tick_positions)
ax2.set_xticklabels(tick_labels, rotation=30, ha='right', fontsize=9)

plt.tight_layout()
plt.show()


## Step 8: 数据统计汇总


In [ ]:
# ============================
# Step 8: 统计汇总
# ============================

print("=" * 60)
print("  光线传媒（300251.SZ）近一年数据统计")
print("=" * 60)
print(f"  数据区间:   {df['date'].min().strftime('%Y-%m-%d')}  ~  {df['date'].max().strftime('%Y-%m-%d')}")
print(f"  交易天数:   {len(df)} 天")
print(f"  期初收盘价: ¥{df.iloc[0]['close']:.2f}")
print(f"  期末收盘价: ¥{df.iloc[-1]['close']:.2f}")
print(f"  区间涨跌幅: {(df.iloc[-1]['close'] / df.iloc[0]['close'] - 1) * 100:+.2f}%")
print("-" * 60)
print(f"  最高价:     ¥{df['high'].max():.2f}  ({df.loc[df['high'].idxmax(), 'date'].strftime('%Y-%m-%d')})")
print(f"  最低价:     ¥{df['low'].min():.2f}  ({df.loc[df['low'].idxmin(), 'date'].strftime('%Y-%m-%d')})")
print(f"  收盘均价:   ¥{df['close'].mean():.2f}")
print(f"  收盘中位数: ¥{df['close'].median():.2f}")
print(f"  标准差:     ¥{df['close'].std():.2f}")
print("-" * 60)
print(f"  日均成交量: {df['volume_wan'].mean():.0f} 万手")
print(f"  最大成交量: {df['volume_wan'].max():.0f} 万手")
print(f"  日均成交额: {df['amount'].mean():.0f} 千元")
print(f"  最大成交额: {df['amount'].max():.0f} 千元")
print("-" * 60)
print(f"  上涨天数:   {(df['pct_change'] > 0).sum()} 天 ({(df['pct_change'] > 0).sum() / len(df) * 100:.1f}%)")
print(f"  下跌天数:   {(df['pct_change'] < 0).sum()} 天 ({(df['pct_change'] < 0).sum() / len(df) * 100:.1f}%)")
print(f"  最大单日涨幅: {df['pct_change'].max():+.2f}%")
print(f"  最大单日跌幅: {df['pct_change'].min():+.2f}%")
print("=" * 60)


## Bonus: 导出为 CSV

将处理后的数据保存为 CSV，方便 Excel 或其他工具打开。


In [ ]:
# ============================
# Bonus: 导出为 CSV
# ============================

output_columns = ['date', 'open', 'high', 'low', 'close', 'volume', 'amount',
                  'pct_change', 'MA5', 'MA10', 'MA20', 'MA60']

df[output_columns].to_csv('光线传媒_300251_分析.csv', index=False, encoding='utf-8-sig')
print("✅ 已导出 CSV: 光线传媒_300251_分析.csv")
print(f"  共 {len(df)} 行 × {len(output_columns)} 列")


---

## 总结

本 Notebook 完成了以下工作：

| 步骤 | 内容 | 工具 / 方法 |
|------|------|------------|
| 1 | 导入依赖库 | pandas, matplotlib, mplfinance, tushare |
| 2 | 数据获取 | Tushare `pro.daily()` / 本地 JSON |
| 3 | 数据清洗 | 缺失值检查、日期转换、特征工程 |
| 4 | 收盘价趋势图 | matplotlib 折线图 + 均线 + 极值标注 |
| 5 | 成交量柱状图 | matplotlib bar（涨红跌绿） |
| 6 | K 线图 | mplfinance candle + volume |
| 7 | 综合看板 | matplotlib 纯手工 K 线 + 成交量联动 |
| 8 | 统计汇总 | pandas describe + 自定义统计 |
| Bonus | CSV 导出 | pandas to_csv (UTF-8 BOM) |

> 💡 键盘操作提示：在 Jupyter 中按 `Shift + Enter` 运行当前 Cell，按 `Ctrl + Enter` 运行后停留。
